In [1]:
import pandas as pd 

## EDA

In [2]:
# load the dataset and make sure the path is correct
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent  # notebooks/ -> project root
data = pd.read_csv(PROJECT_ROOT / 'data' / 'raw' / 'WA_Fn-UseC_-Telco-Customer-Churn.csv')

In [3]:
data.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [4]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


In [5]:
# look at churn data first as thats the most important feature for our model.
data['Churn'].value_counts()



Churn
No     5174
Yes    1869
Name: count, dtype: int64

In [6]:
data['Churn'].value_counts(normalize= True)

Churn
No     0.73463
Yes    0.26537
Name: proportion, dtype: float64

In [7]:
# looking at total charges and converting it to numeric as it is object type. 
# it did raise an error can uncomment code to check when rerunning
# pd.to_numeric(data['TotalCharges'],errors = 'raise')

In [8]:
# check how many rows of total charges are effected by the conversion issue
pd.to_numeric(data['TotalCharges'], errors='coerce').isna().sum()

np.int64(11)

In [9]:
# check if the blanks have a tneure of 0 months
data[pd.to_numeric(data['TotalCharges'], errors='coerce').isna()][['tenure', 'TotalCharges', 'MonthlyCharges']]

,tenure,TotalCharges,MonthlyCharges
488,0,,52.55
753,0,,20.25
936,0,,80.85
1082,0,,25.75
1340,0,,56.05
3331,0,,19.85
3826,0,,25.35
4380,0,,20.00
5218,0,,19.70
6670,0,,73.35


In [10]:
# so now we no that these customers have in fact not paid anything yet we need to convert them to 0 as we need to record a numeric value of them paying nothing.
data['TotalCharges'] = pd.to_numeric(data['TotalCharges'], errors='coerce').fillna(0)
data['TotalCharges'].isna().sum()  # should be 0 now
data['TotalCharges'].dtype          # should be float64, not object

dtype('float64')

In [11]:
# customerID is a unique identifier, not a predictive feature — drop it
data = data.drop('customerID', axis=1)

In [12]:

# duplicate check
data.duplicated().sum()

np.int64(22)

In [13]:
# churn rate by contract type
data.groupby('Contract')['Churn'].value_counts(normalize=True)

Contract        Churn
Month-to-month  No       0.572903
                Yes      0.427097
One year        No       0.887305
                Yes      0.112695
Two year        No       0.971681
                Yes      0.028319
Name: proportion, dtype: float64

In [14]:
# tenure distribution by churn status
data.groupby('Churn')['tenure'].describe()

,count,mean,std,min,25%,50%,75%,max
Churn,,,,,,,,
No,5174.0,37.569965,24.113777,0.0,15.0,38.0,61.0,72.0
Yes,1869.0,17.979133,19.531123,1.0,2.0,10.0,29.0,72.0


In [15]:
# churn rate by internet service type
data.groupby('InternetService')['Churn'].value_counts(normalize=True)

InternetService  Churn
DSL              No       0.810409
                 Yes      0.189591
Fiber optic      No       0.581072
                 Yes      0.418928
No               No       0.925950
                 Yes      0.074050
Name: proportion, dtype: float64

In [16]:
# churn rate by payment method
data.groupby('PaymentMethod')['Churn'].value_counts(normalize=True)

PaymentMethod              Churn
Bank transfer (automatic)  No       0.832902
                           Yes      0.167098
Credit card (automatic)    No       0.847569
                           Yes      0.152431
Electronic check           No       0.547146
                           Yes      0.452854
Mailed check               No       0.808933
                           Yes      0.191067
Name: proportion, dtype: float64

In [17]:
# check the "No internet service" redundant-category quirk
for col in ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies']:
    print(col, data[col].unique())

OnlineSecurity ['No' 'Yes' 'No internet service']
OnlineBackup ['Yes' 'No' 'No internet service']
DeviceProtection ['No' 'Yes' 'No internet service']
TechSupport ['No' 'Yes' 'No internet service']
StreamingTV ['No' 'Yes' 'No internet service']
StreamingMovies ['No' 'Yes' 'No internet service']


In [18]:
# cardinality of every categorical column — informs encoding strategy
data.select_dtypes(include='object').nunique()

gender              2
Partner             2
Dependents          2
PhoneService        2
MultipleLines       3
InternetService     3
OnlineSecurity      3
OnlineBackup        3
DeviceProtection    3
TechSupport         3
StreamingTV         3
StreamingMovies     3
Contract            3
PaperlessBilling    2
PaymentMethod       4
Churn               2
dtype: int64

In [19]:
# check multicollinearity between the numeric charge/tenure columns
data[['tenure', 'MonthlyCharges', 'TotalCharges']].corr()

,tenure,MonthlyCharges,TotalCharges
tenure,1.000000,0.247900,0.826178
MonthlyCharges,0.247900,1.000000,0.651174
TotalCharges,0.826178,0.651174,1.000000


## Preproccesing 

In [21]:
# dropping TotalCharges as it is highly correlated with MonthlyCharges and tenure
# sanity check to make sure colomn is dropped
data = data.drop('TotalCharges', axis=1)
print(data.columns.tolist())


['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'Churn']


In [24]:
# so a few colomns have a third category saying no sevrice, we can convert then to no so when we encode it it doesnt add any dumby info 
cols_with_dependency = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies']
for cols in cols_with_dependency:
    data[cols] = data[cols].replace({'No internet service': 'No'})
# multiplelines say no phone service (differen to the other cols), we can convert it to no so when we encode it it doesnt add any dumby info
data['MultipleLines'] = data['MultipleLines'].replace('No phone service', 'No')
# sanity check to make sure the replacement worked
for cols in cols_with_dependency + ['MultipleLines']: 
    print(cols, data[cols].unique())

OnlineSecurity ['No' 'Yes']
OnlineBackup ['Yes' 'No']
DeviceProtection ['No' 'Yes']
TechSupport ['No' 'Yes']
StreamingTV ['No' 'Yes']
StreamingMovies ['No' 'Yes']
MultipleLines ['No' 'Yes']


In [25]:
# now we cleaned the data we now need to encode the data so it can be processed by themodel, we are most likely using a one hot encoding as we are looking at a logistic regression problem , to avoid non colinearity we drop a layer first. 
X = pd.get_dummies(data.drop('Churn', axis=1), drop_first=True)
y = y = data['Churn']

# sanity check to make sure encoding worked
print(X.columns.tolist())
print(X.shape)

['SeniorCitizen', 'tenure', 'MonthlyCharges', 'gender_Male', 'Partner_Yes', 'Dependents_Yes', 'PhoneService_Yes', 'MultipleLines_Yes', 'InternetService_Fiber optic', 'InternetService_No', 'OnlineSecurity_Yes', 'OnlineBackup_Yes', 'DeviceProtection_Yes', 'TechSupport_Yes', 'StreamingTV_Yes', 'StreamingMovies_Yes', 'Contract_One year', 'Contract_Two year', 'PaperlessBilling_Yes', 'PaymentMethod_Credit card (automatic)', 'PaymentMethod_Electronic check', 'PaymentMethod_Mailed check']
(7043, 22)


In [ ]:
# stratified train-test split to maintain churn distribution in both sets with random state to keep the split reproducible
from sklearn.model_selection import train_test_split


X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# sanity check to make sure the split worked and they are similar in shape and distribution
print(X_train.shape, X_test.shape)
print(y_train.value_counts(normalize=True), y_test.value_counts(normalize=True))

(5634, 22) (1409, 22)
Churn
No     0.734647
Yes    0.265353
Name: proportion, dtype: float64 Churn
No     0.734564
Yes    0.265436
Name: proportion, dtype: float64


In [29]:
# scale the features of numerical nature to mean 0 and standard deviation 1, and obviously fit onto the training set to aviod data leakage, and then transform the test set with the same scaler.

from sklearn.preprocessing import StandardScaler
numeric_cols = ['tenure', 'MonthlyCharges']
scaler = StandardScaler()
X_train[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])  
# scale the test set so both are run on the same scale and avoid data leakage
X_test[numeric_cols] = scaler.transform(X_test[numeric_cols]) 

# sanity check to make sure the scaling worked
print(X_train[numeric_cols].describe())



             tenure  MonthlyCharges
count  5.634000e+03    5.634000e+03
mean  -1.008935e-17   -2.402527e-16
std    1.000089e+00    1.000089e+00
min   -1.322329e+00   -1.544028e+00
25%   -9.559779e-01   -9.711977e-01
50%   -1.418632e-01    1.848336e-01
75%    9.164859e-01    8.319124e-01
max    1.608483e+00    1.785939e+00


In [ ]:
# train model using logistic regression with class weight balanced to account for the imbalance in the churn data, and a random state for reproducibility, and max_iter to ensure convergence.
from sklearn.linear_model import LogisticRegression
model = LogisticRegression(class_weight='balanced', random_state=42, max_iter=1000)
model.fit(X_train, y_train)

print("Model trained.")

Model trained.


In [ ]:
# run the model on the test set and get the predictions, then we can evaluate the model using classification report and confusion matrix that can tell us more about how the model performed.
from sklearn.metrics import classification_report, confusion_matrix

y_pred = model.predict(X_test)

In [33]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

          No       0.90      0.73      0.81      1035
         Yes       0.51      0.78      0.61       374

    accuracy                           0.74      1409
   macro avg       0.70      0.75      0.71      1409
weighted avg       0.80      0.74      0.75      1409



In [34]:
print(confusion_matrix(y_test, y_pred))

[[754 281]
 [ 84 290]]
